In [3]:
import cv2
import numpy as np
import os

# 1. Define your Taxonomy
CLASSES = {
    1: ("SHAFT", (0, 0, 255)),       2: ("CROSSARM", (0, 255, 255)),
    3: ("PRIMARY", (0, 255, 0)),     4: ("SECONDARY", (0, 180, 0)),
    5: ("NEUTRAL", (255, 255, 0)),   6: ("GUY", (255, 0, 255)),
    7: ("ANCHOR", (0, 165, 255)),    8: ("TRANSFORMER", (255, 0, 0))
}

def semantic_labeler(img_path, save_path):
    img_orig = cv2.imread(img_path)
    if img_orig is None: return True
    
    img = img_orig.copy()
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    
    # History stacks for Undo
    img_history = [img.copy()]
    mask_history = [mask.copy()]
    
    active_id = 1
    brush_size = 5
    drawing = False

    def paint(event, x, y, flags, param):
        nonlocal drawing, brush_size
        if event == cv2.EVENT_LBUTTONDOWN:
            # Save state BEFORE drawing starts
            img_history.append(img.copy())
            mask_history.append(mask.copy())
            drawing = True
            if len(img_history) > 30: # Limit history to save RAM
                img_history.pop(0)
                mask_history.pop(0)
                
        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False
            
        elif event == cv2.EVENT_MOUSEMOVE and drawing:
            # Draw on both visual and semantic layers
            cv2.circle(mask, (x, y), brush_size, active_id, -1)
            cv2.circle(img, (x, y), brush_size, CLASSES[active_id][1], -1)
            
        # --- UNDO LOGIC: Right-Click to Undo ---
        elif event == cv2.EVENT_RBUTTONDOWN:
            if len(img_history) > 1:
                img[:] = img_history.pop()
                mask[:] = mask_history.pop()

    cv2.namedWindow("Poux Workflow - Labeler")
    cv2.setMouseCallback("Poux Workflow - Labeler", paint)

    print(f"Now Labeling: {os.path.basename(img_path)}")
    print("CONTROLS: 1-8: Classes | [ ]: Brush Size | Right-Click: Undo | S: Save | Q: Quit")

    while True:
        display = img.copy()
        # Visual feedback for class and brush size
        cv2.putText(display, f"Class: {CLASSES[active_id][0]} | Brush: {brush_size}px", 
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.imshow("Poux Workflow - Labeler", display)
        
        k = cv2.waitKey(1) & 0xFF
        if ord('1') <= k <= ord('8'): active_id = int(chr(k))
        elif k == ord('['): brush_size = max(1, brush_size - 1)
        elif k == ord(']'): brush_size = min(50, brush_size + 1)
        elif k == ord('s'):
            cv2.imwrite(save_path, mask)
            break
        elif k == ord('q'): 
            cv2.destroyAllWindows()
            return False
    
    cv2.destroyAllWindows()
    return True

# 2. Set your paths
image_dir = r"D:\Intern\Lidar\pole_extraction\pole_images"
mask_dir = r"D:\Intern\Lidar\pole_extraction\masks"
if not os.path.exists(mask_dir): os.makedirs(mask_dir)

# 3. Main Processing Loop
for file in os.listdir(image_dir):
    if file.lower().endswith(('.jpg', '.png', '.jpeg')):
        mask_filename = file.replace(".jpg", "_mask.png").replace(".jpeg", "_mask.png").replace(".png", "_mask.png")
        save_path = os.path.join(mask_dir, mask_filename)
        
        if os.path.exists(save_path):
            continue # Already done
            
        full_path = os.path.join(image_dir, file)
        if not semantic_labeler(full_path, save_path):
            print("Quitting script...")
            break

Loading LAS file...
Cropping Pole 0...
Success! Extracted 920 points for Pole 0.
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The handle is invalid. 
[Open3D WARNING] [ViewControl] SetViewPoint() failed because window height and width are not set.


In [4]:
import open3d as o3d
import laspy
import numpy as np
import cv2
import os
import pandas as pd
from sklearn.decomposition import PCA

# 1. Setup all your paths
las_file = r"D:\Intern\Lidar\pole_extraction\poles_with_crossarms_qgis1.las"
csv_file = r"D:\Intern\Lidar\pole_extraction\pole_locations.csv"
mask_dir = r"D:\Intern\Lidar\pole_extraction\masks"
output_las = r"D:\Intern\Lidar\pole_extraction\fully_labeled_infrastructure.las"
print("Unique classes in file:", np.unique(data.classification))
# 2. Load the 3D World
print("Loading original LAS file...")
las = laspy.read(las_file)
points_3d = np.vstack((las.x, las.y, las.z)).transpose()

# Create a master label array for the entire point cloud (Default 0 = Background)
master_labels = np.zeros(len(points_3d), dtype=np.uint8)

# 3. Load the CSV Data
df = pd.read_csv(csv_file)
search_radius = 1.2 # Tight crop to avoid background trees

print(f"Starting batch projection for {len(df)} poles...")

# 4. The Automation Loop
success_count = 0

for index, row in df.iterrows():
    pole_id = int(row['Pole_ID'])
    pole_x = row['X']
    pole_y = row['Y']
    
    # Construct the expected mask filename
    # NOTE: Change this if your naming convention is different!
    mask_name = f"pole_{pole_id}_mask.png" 
    mask_path = os.path.join(mask_dir, mask_name)
    
    if not os.path.exists(mask_path):
        print(f"Skipping Pole {pole_id}: Could not find {mask_name}")
        continue
        
    # Load the Mask
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None: continue
    mask_h, mask_w = mask.shape
    
    # Extract the 3D points for THIS specific pole
    distances = np.sqrt((points_3d[:, 0] - pole_x)**2 + (points_3d[:, 1] - pole_y)**2)
    pole_indices = np.where(distances <= search_radius)[0]
    
    if len(pole_indices) < 50:
        print(f"Skipping Pole {pole_id}: Not enough 3D points found.")
        continue

    isolated_points = points_3d[pole_indices]
    
    # --- PCA Math (Same as before) ---
    xy_points = isolated_points[:, :2]
    pca = PCA(n_components=2)
    pca.fit(xy_points)
    u_coords = pca.transform(xy_points)[:, 0] 
    
    z_coords = isolated_points[:, 2]
    
    z_min, z_max = z_coords.min(), z_coords.max()
    v_norm = 1.0 - ((z_coords - z_min) / (z_max - z_min)) 
    v_pixels = np.clip((v_norm * (mask_h - 1)).astype(int), 0, mask_h - 1)
    
    u_min, u_max = u_coords.min(), u_coords.max()
    u_norm = (u_coords - u_min) / (u_max - u_min)
    u_pixels = np.clip((u_norm * (mask_w - 1)).astype(int), 0, mask_w - 1)
    
    # --- Inject Labels into Master Array ---
    for i in range(len(isolated_points)):
        global_idx = pole_indices[i] # The index in the massive 32M point cloud
        u = u_pixels[i]
        v = v_pixels[i]
        
        class_id = mask[v, u]
        
        # Only overwrite if the class is NOT background (0)
        # This prevents accidental overwriting of overlapping poles
        if class_id != 0:
            master_labels[global_idx] = class_id
            
    print(f"Successfully projected labels for Pole {pole_id}")
    success_count += 1

# 5. Save the final Labeled LAS File
print(f"\nFinished processing. {success_count} masks applied.")
print("Saving final labeled LAS file...")

# We inject your new classes into the standard LAS "classification" field
las.classification = master_labels
las.write(output_las)

print(f"SUCCESS! Your labeled dataset is saved at:\n{output_las}")

Loading original LAS file...
Starting batch projection for 64 poles...
Successfully projected labels for Pole 0
Successfully projected labels for Pole 1
Skipping Pole 2: Could not find pole_2_mask.png
Successfully projected labels for Pole 3
Successfully projected labels for Pole 4
Successfully projected labels for Pole 5
Successfully projected labels for Pole 6
Skipping Pole 7: Could not find pole_7_mask.png
Skipping Pole 8: Could not find pole_8_mask.png
Successfully projected labels for Pole 9
Successfully projected labels for Pole 10
Successfully projected labels for Pole 11
Successfully projected labels for Pole 12
Successfully projected labels for Pole 13
Successfully projected labels for Pole 14
Successfully projected labels for Pole 15
Successfully projected labels for Pole 16
Successfully projected labels for Pole 17
Skipping Pole 18: Could not find pole_18_mask.png
Successfully projected labels for Pole 19
Skipping Pole 20: Could not find pole_20_mask.png
Skipping Pole 21: Co

In [1]:
import laspy, numpy as np
print("Classes found in file:", np.unique(laspy.read(r"D:\Intern\Lidar\pole_extraction\fully_labeled_infrastructure.las").classification))

Classes found in file: [0 1 2 3 4 5 6 7 8]


In [1]:
import laspy
import numpy as np
import cv2
import os
import pandas as pd
from sklearn.decomposition import PCA

# 1. Paths
las_input = r"D:\Intern\Lidar\pole_extraction\poles_with_crossarms_qgis1.las"
csv_file = r"D:\Intern\Lidar\pole_extraction\pole_locations.csv"
mask_dir = r"D:\Intern\Lidar\pole_extraction\masks"
output_las = r"D:\Intern\Lidar\pole_extraction\fully_labeled_v2.las"

# 2. Load and Reset
las = laspy.read(las_input)
points_3d = np.vstack((las.x, las.y, las.z)).transpose()

# SET EVERYTHING TO 0 (Background/Wires) so they don't disappear
master_labels = np.zeros(len(points_3d), dtype=np.uint8)

df = pd.read_csv(csv_file)
search_radius = 1.2 

print("Projecting labels to User-Defined range (64-72)...")

for index, row in df.iterrows():
    mask_name = f"pole_{int(row['Pole_ID'])}_mask.png"
    mask_path = os.path.join(mask_dir, mask_name)
    if not os.path.exists(mask_path): continue
        
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask_h, mask_w = mask.shape
    
    # Crop logic
    dist = np.sqrt((points_3d[:, 0] - row['X'])**2 + (points_3d[:, 1] - row['Y'])**2)
    indices = np.where(dist <= search_radius)[0]
    if len(indices) < 10: continue
    
    pts = points_3d[indices]
    pca = PCA(n_components=2).fit(pts[:, :2])
    u_coords = pca.transform(pts[:, :2])[:, 0]
    
    # Normalize
    z_min, z_max = pts[:, 2].min(), pts[:, 2].max()
    v_px = ((1.0 - (pts[:, 2] - z_min) / (z_max - z_min)) * (mask_h - 1)).astype(int)
    u_min, u_max = u_coords.min(), u_coords.max()
    u_px = (((u_coords - u_min) / (u_max - u_min)) * (mask_w - 1)).astype(int)
    
    for i in range(len(indices)):
        class_id = mask[v_px[i], u_px[i]]
        if class_id > 0:
            # SHIFTING: Add 64 to move to the "User Defined" LAS range
            master_labels[indices[i]] = class_id + 64
            
    print(f"Pole {int(row['Pole_ID'])} projected.")

las.classification = master_labels
las.write(output_las)
print(f"Success! Use {output_las} in QGIS.")

Projecting labels to User-Defined range (64-72)...
Pole 0 projected.
Pole 1 projected.
Pole 3 projected.
Pole 4 projected.
Pole 5 projected.
Pole 6 projected.
Pole 9 projected.
Pole 10 projected.
Pole 11 projected.
Pole 12 projected.
Pole 13 projected.
Pole 14 projected.
Pole 15 projected.
Pole 16 projected.
Pole 17 projected.
Pole 19 projected.
Pole 22 projected.
Pole 23 projected.
Pole 24 projected.
Pole 25 projected.
Pole 27 projected.
Pole 28 projected.
Pole 29 projected.
Pole 30 projected.
Pole 31 projected.
Pole 32 projected.
Pole 33 projected.
Pole 36 projected.
Pole 37 projected.
Pole 38 projected.
Pole 40 projected.
Pole 41 projected.
Pole 42 projected.
Pole 43 projected.
Pole 44 projected.
Pole 45 projected.
Pole 46 projected.
Success! Use D:\Intern\Lidar\pole_extraction\fully_labeled_v2.las in QGIS.
